In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
import os

llm = ChatOpenAI(
    model="google/gemma-4-31b-it:free",
    base_url="https://openrouter.ai/api/v1",
    #api_key="",
    model_kwargs={
        "reasoning": {"enabled": True},
    },
)

# First call
messages = [
    HumanMessage(content="How many r's are in the word 'strawberry'?")
]

response = llm.invoke(messages)

print(response.content)

# Extract reasoning_details from the raw response
reasoning_details = (
    response.response_metadata
    .get("reasoning_details")
)

# Preserve reasoning_details for the next turn
messages = [
    HumanMessage(content="How many r's are in the word 'strawberry'?"),
    AIMessage(
        content=response.content,
        additional_kwargs={
            "reasoning_details": reasoning_details
        },
    ),
    HumanMessage(content="Are you sure? Think carefully.")
]

response2 = llm.invoke(messages)

print(response2.content)

/home/praveen/Desktop/Post_Generator/.venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3688: UserWarning: Parameters {'reasoning'} should be specified explicitly. Instead they were passed in as part of `model_kwargs` parameter.
  if await self.run_code(code, result, async_=asy):


[{'id': 'rs_tmp_bfoapdzvem', 'summary': [], 'type': 'reasoning', 'content': [{'text': 'The user is asking for the count of the letter \'r\' in the word "strawberry".\n\n    *   Word: "strawberry"\n    *   Characters: s, t, r, a, w, b, e, r, r, y\n\n    *   s (no)\n    *   t (no)\n    *   r (1)\n    *   a (no)\n    *   w (no)\n    *   b (no)\n    *   e (no)\n    *   r (2)\n    *   r (3)\n    *   y (no)\n\nThere are 3 \'r\'s.', 'type': 'reasoning_text'}], 'status': 'completed', 'format': 'unknown'}, {'type': 'text', 'text': "There are 3 r's in strawberry.", 'annotations': [], 'id': 'msg_tmp_n4n1bdrnt9'}]
[{'id': 'rs_tmp_gwl6dwai0xt', 'summary': [], 'type': 'reasoning', 'content': [{'text': 'The user is asking me to double-check the number of \'r\'s in the word "strawberry".\n\n    *   Word: "strawberry"\n    *   Letters: s, t, r, a, w, b, e, r, r, y\n\n    *   1st \'r\': st**r**awberry\n    *   2nd \'r\': strawbe**r**ry\n    *   3rd \'r\': strawber**r**y\n\n    *   There are exactly 3 \'

In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

from ai.schema.model import Article

load_dotenv()

# api_key = os.getenv("GOOGLE_API_KEY")

# if not api_key:
#     raise ValueError("GOOGLE_API_KEY not found in environment variables")


class ContentGenerator:
    def __init__(
        self,
        model: str = "google/gemma-4-31b-it:free",
    ):
        self.client = ChatOpenAI(
            model=model,
            api_key = os.getenv("OPENROUTER_API_KEY")
            base_url="https://openrouter.ai/api/v1",
            temperature=0.2,
        )

        self.structured_llm = self.client.with_structured_output(Article)

    def generate(self, idea: str) -> Article:
        system_prompt = """
You are an expert technical writer.

The user provides a topic or short idea.

Generate a high-quality educational article.

Requirements:
- Generate an engaging title.
- Write a detailed educational article.
- 700-1200 words.
- Organize the article with clear headings.
- Explain concepts with practical examples.
- Be technically accurate.
- Do not use markdown.
- Do not use bullet lists unless absolutely necessary.
- Do not include citations.
- Do not include phrases like:
  "Here is your article."

Return ONLY the structured response.
"""

        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=idea),
        ]

        article: Article = self.structured_llm.invoke(messages)

        return article


if __name__ == "__main__":

    generator = ContentGenerator()

    article = generator.generate(
        "Explain Model Context Protocol (MCP) and its benefits with real-world examples."
    )

    print("=" * 80)
    print("TITLE")
    print("=" * 80)
    print(article.title)

    print("\n")

    print("=" * 80)
    print("CONTENT")
    print("=" * 80)
    print(article.content)

TypeError: 'NoneType' object is not iterable